In [1]:
import os
import time
import json
import pickle
import pandas as pd
import numpy as np
from functools import partial
import joblib

from datetime import datetime
from tqdm import tqdm
from dotenv import load_dotenv
from pathlib import Path

In [2]:
from text2graphapi.src.IntegratedSyntacticGraph import ISG
import networkx as nx
from collections import Counter

import torch

import optuna
import mlflow
from databricks.sdk import WorkspaceClient

[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jovyan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


2025-12-20 16:32:32,819; - INFO; - generated new fontManager
2025-12-20 16:32:33,189; - DEBUG; - Import libraries/modules from :PROD


In [3]:
import seaborn as sns
import matplotlib.pyplot as plt

Define path variables

In [4]:
representation_type = "integrated_syntactic_graph"
developer_initials = "JP"

In [5]:
current_dir = Path.cwd()
env_path = current_dir.parent.parent / "conf" / "local" / ".env"
results_path = current_dir.parent.parent / "results" / "graph"

train_data_full_cleaned_path = current_dir.parent.parent / "data" /  "01_processed" / "pan20-authorship-verification-training-large-cleaned.jsonl"
validation_data_full_cleaned_path = current_dir.parent.parent / "data" /  "01_processed" / "pan20-authorship-verification-validation-large-cleaned.jsonl"
test_data_full_cleaned_path = current_dir.parent.parent / "data" /  "01_processed" / "pan21-authorship-verification-test-cleaned.jsonl"

Connect to databricks for logging results

In [6]:
load_dotenv(env_path)

w = WorkspaceClient()   
print("Connected to:", w.config.host)

mlflow.set_tracking_uri("databricks")
mlflow.autolog()

2025/12/20 16:32:34 WARNING mlflow.utils.autologging_utils: MLflow sklearn autologging is known to be compatible with 1.4.0 <= scikit-learn, but the installed version is 1.3.2. If you encounter errors during autologging, try upgrading / downgrading scikit-learn to a compatible version, or try upgrading MLflow.


Connected to: https://dbc-1ea3ad0e-f504.cloud.databricks.com


2025/12/20 16:32:34 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2025/12/20 16:32:34 WARNING mlflow.utils.autologging_utils: MLflow statsmodels autologging is known to be compatible with 0.14.1 <= statsmodels, but the installed version is 0.14.0. If you encounter errors during autologging, try upgrading / downgrading statsmodels to a compatible version, or try upgrading MLflow.
2025/12/20 16:32:35 INFO mlflow.tracking.fluent: Autologging successfully enabled for statsmodels.


What are GPU are the experiments run on

In [7]:
!nvidia-smi

Sat Dec 20 16:32:36 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100 80GB PCIe          Off |   00000000:21:00.0 Off |                    0 |
| N/A   52C    P0             53W /  300W |       4MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [8]:
running_on_gpu = torch.cuda.is_available()

In [9]:
if running_on_gpu:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_props = torch.cuda.get_device_properties(0)
    gpu_vram_gb = round(gpu_props.total_memory / (1024**3), 2)
else:
    gpu_name = "CPU"
    gpu_props = "N/A"
    gpu_vram_gb = 0

Empty the GPU from previous experiments

In [10]:
import gc
import torch
torch.cuda.empty_cache()
gc.collect()

69

In [11]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

Classification threshold constant specification

In [12]:
classification_thresholds = [x/1000 for x in range(200, 999)]

# Load dataset

#### Load training data

In [13]:
train_data_file_size = os.path.getsize(train_data_full_cleaned_path)
train_data = []

with open(train_data_full_cleaned_path, 'r') as f:
    with tqdm(total=train_data_file_size, desc="Loading data", unit='B', unit_scale=True) as pbar:
        for line in f:
            train_data.append(json.loads(line))
            pbar.update(len(line.encode('utf-8')))

print(f"\nSuccessfully loaded {len(train_data)} items.")

Loading data: 100%|██████████| 11.3G/11.3G [02:00<00:00, 93.8MB/s]


Successfully loaded 273301 items.


In [14]:
train_data_df = pd.DataFrame(train_data)

Prepare dataset for cosine embedding loss

In [15]:
train_data_df.head(10)

,id,pair,same
0,e05b9c0b-88a1-5608-b7e8-ab1fc6b78dc1,"[Well, ever since you and Kurt broke up youve ...",False
1,12f73a20-cdf3-58df-b5bb-9392eec9b486,"[The thing is, Ryouga has no reason to run aft...",False
2,d82c6764-451b-544c-8711-c139e9349c56,"[Ehhhh nah, its silly' Its my job to listen to...",True
3,876b8380-9260-5427-93e8-dc31155c3edd,"[Glaring at the arrogant spark, Always asks va...",False
4,357e8471-35b9-50b4-9ac0-286ac0e8b101,[Runa limped across the small space to an open...,False
5,6b39fe22-409f-5329-9fe1-ddf4a70bedd1,"[And thats retired Commander, if you please St...",False
6,76ed2017-0c9f-580f-83b1-5a041a8169ec,[Meet you downstairs in twenty minutes I say w...,True
7,fd8deb2e-06de-5c92-9a4c-927891c53657,"[Since Ive seen so many others do so, Im going...",False
8,3ce5e811-a57c-5fbf-9f9a-2ee636e50be6,[party Eishi exclaimed Omi shook his head and ...,False
9,25a17cd2-6b01-5fba-99ef-e631e56e181d,[After a few moments she found that Red was ri...,False


#### Create a subset of training data for finetuning

In [16]:
train_tuning_size = 999
train_tuning_data_df = train_data_df.sample(n=train_tuning_size, random_state=42)
train_tuning_data_df = train_tuning_data_df.reset_index(drop=True)
train_tuning_data_df.head()

,id,pair,same
0,1c4a05b6-dabb-5a9d-9e6a-7709b07d8dff,[comes home at about onethirty that night Case...,True
1,aff2cdfb-625d-5fa7-acb8-a0ff9876b34f,"[He rubbed his tired, gritty eyes The clock on...",False
2,15fc44d2-e1c9-574b-9fa3-fdb29f20cc68,"[Mikado grinned and explained, no longer using...",False
3,e77681c2-8d90-5cac-8ab0-07d7c9277670,[Thanks he smiled Now to write Characters and ...,False
4,8c38b587-f54c-5e52-94cb-abb48ce14c5a,"[soldierscare that ended earlier, and all were...",True


#### Load validation data

In [17]:
val_data_file_size = os.path.getsize(validation_data_full_cleaned_path)
val_data = []

with open(validation_data_full_cleaned_path, 'r') as f:
    with tqdm(total=val_data_file_size, desc="Loading data", unit='B', unit_scale=True) as pbar:
        for line in f:
            val_data.append(json.loads(line))
            pbar.update(len(line.encode('utf-8')))

print(f"\nSuccessfully loaded {len(val_data)} items.")

Loading data: 100%|██████████| 103M/103M [00:01<00:00, 97.3MB/s] 


Successfully loaded 2500 items.


In [18]:
val_data_df = pd.DataFrame(val_data)

#### Load testing data

In [19]:
test_data_file_size = os.path.getsize(test_data_full_cleaned_path)
test_data = []

with open(test_data_full_cleaned_path, 'r') as f:
    with tqdm(total=test_data_file_size, desc="Loading data", unit='B', unit_scale=True) as pbar:
        for line in f:
            test_data.append(json.loads(line))
            pbar.update(len(line.encode('utf-8')))

print(f"\nSuccessfully loaded {len(test_data)} items.")

Loading data: 100%|██████████| 826M/826M [00:09<00:00, 82.7MB/s] 


Successfully loaded 19999 items.


In [20]:
test_data_df = pd.DataFrame(test_data)

In [21]:
train_tuning_data_df = train_tuning_data_df.head()
val_data_df = val_data_df.head()

# Functions to build graphs and extract features

Format the texts in format required by text2graphapi

In [215]:
def texts_to_isg_format(isg_object, texts):
    corpus_docs = [
        {"id": i, "doc": text}
        for i, text in enumerate(texts)
    ]
    return isg_object.transform(corpus_docs)

Parse ISG's nodes POS and lemma function

In [153]:
def parse_graph_node(graph_node):
    node_str = str(graph_node)
    if "_" in node_str:
        lemma, pos = node_str.rsplit("_", 1)
        return lemma.lower(), pos

Parse dependency

In [154]:
def parse_graph_dependency(data):
    dependency = data.get("gramm_relation")
    parsed_dependency = dependency.split("_", 1)[0]
    return parsed_dependency

Extract multi-level features from graph

In [214]:
def extract_features_from_isg(graph):
    graph_object = graph["graph"]
    features = Counter()

    for node in graph_object.nodes:
        lemma, pos = parse_node(node)
        features[f"LEX::{lemma}"] += 1
        
        if pos:
            features[f"POS::{pos}"] += 1

    for _, _, data in graph_object.edges(data=True):
        dependency = parse_dependency(data)
        if dependency:
            features[f"DEP::{dependency}"] += 1

    return features

Build vocabulary from counters function

In [ ]:
def build_vocab(counters):
    vocab = sorted(set().union(*counters))
    index = {f: i for i, f in enumerate(vocab)}
    return vocab, index

Build vectors based on vocabulary

In [ ]:
def build_vector(counter, index):
    vector = np.zeros(len(index), dtype=np.float32)
    for feature, value in counter.items():
        if feature in index:
            vector[index[feature]] = value
    return vector

Convert texts to text2graphapi integrated syntactic graphs

In [216]:
def build_isg_features(train_df, test_df):
    
    isg = ISG(
        graph_type="DiGraph",
        language="en",
        apply_prep=True,
        output_format="networkx"
    )
    
    train_texts1 = train_df["pair"].apply(lambda x: x[0])
    train_texts2 = train_df["pair"].apply(lambda x: x[1])
    test_texts1 = test_df["pair"].apply(lambda x: x[0])
    test_texts2 = test_df["pair"].apply(lambda x: x[1])
    
    print("Converting to graphs \n")
    X_train1 = texts_to_isg(isg, train_texts1)
    X_train2 = texts_to_isg(isg, train_texts2)
    X_test1 = texts_to_isg(isg, test_texts1)
    X_test2 = texts_to_isg(isg, test_texts2)

    print("Extract features from the graph \n")
    train_features1 = [extract_multilevel_features_from_isg(g) for g in X_train1]
    train_features1 = [extract_multilevel_features_from_isg(g) for g in X_train1]

    end = time.perf_counter()
    print(f"Execution time: {end - start:.6f} seconds")
    
    X_train2 = texts_to_isg(isg, train_texts2)
    
    X_test1 = texts_to_isg(isg, test_texts1)
    X_test2 = texts_to_isg(isg, test_texts2)
    

    X_train = (X_train1, X_train2)
    X_test = (X_test1, X_test2)
    
    return X_train, X_test

# Build graphs and export features

# Export data